# 05 Material properties: the database, new materials, and what the numbers mean

| | |
|---|---|
| **Purpose** | How to load materials from the built-in database, define new ones, save them to the user database, and derive wavelength, wavenumber, attenuation and impedance from them. No BEM solve. |
| **Inputs** | Database materials water, fat and cortical bone; two user-defined materials; frequency 500 kHz for the derived quantities. |
| **Outputs** | Tables of properties; a bar chart of acoustic impedance (`outputs/05_material_properties_1.png`). |
| **Checks** | Loaded values match the database table; wavelength, wavenumber and power-law attenuation follow their definitions; normal-incidence reflection and transmission conserve energy; saving a material works and is undone; a duplicate name raises `ValueError`. |
| **Runtime** | Under 5 s (Docker image, 9 Nov 2026). CI mode is identical. |
| **Adapted from** | `notebooks/Material properties.ipynb` (upstream OptimUS). Unlike the upstream notebook, this one restores the user database afterwards, so it can be rerun and run in CI. |

## Physics notes

Each homogeneous domain is described by its density $\rho$, speed of sound $c$ and a power-law
attenuation $\alpha(f) = a \, (f / 1\,\text{MHz})^b$ in Np/m. OptimUS uses the complex wavenumber
$k = 2\pi f / c + i\alpha(f)$ and the wavelength $\lambda = c / f$.

The characteristic impedance $Z = \rho c$ decides how much energy crosses an interface. At normal
incidence from medium 1 into medium 2 the pressure reflection and transmission coefficients are
(Kinsler et al., *Fundamentals of Acoustics*, 4th ed., 2000, section 6.2)

$$ R = \frac{Z_2 - Z_1}{Z_2 + Z_1}, \qquad T = 1 + R = \frac{2 Z_2}{Z_2 + Z_1}, $$

and the intensity coefficients satisfy $R^2 + \frac{Z_1}{Z_2} T^2 = 1$ (energy is conserved).

> **To expand (Yash):** relate the impedance contrast of water and bone to the reflections you see in 01_basic_model; explain why air inside tissue (06_nested_domains) is almost a perfect reflector.

## Setup

Imports, the `CI_MODE` switch and the output folder.

In [ ]:
NOTEBOOK_NAME = "05_material_properties"

%matplotlib inline
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import optimus

# CI_MODE uses smaller settings so the notebook finishes in a few minutes in CI.
# Set it from the shell with OPTIMUS_CI_MODE=1, or change it here by hand.
CI_MODE = os.environ.get("OPTIMUS_CI_MODE", "0") == "1"

# Figures and a small JSON file of key results are written here.
OUTPUT_DIR = os.environ.get("OPTIMUS_OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS = {"notebook": NOTEBOOK_NAME, "ci_mode": CI_MODE,
           "optimus_version": optimus.__version__, "checks": {}}
START_TIME = time.time()

print("OptimUS", optimus.__version__, "| CI mode:", CI_MODE, "| outputs in:", OUTPUT_DIR)

In [ ]:
def check(name, condition, detail):
    """Record a named check and stop the notebook if it fails."""
    RESULTS["checks"][name] = bool(condition)
    print(("PASS  " if condition else "FAIL  ") + name + ": " + detail)
    assert condition, name + " failed: " + detail


def gmres_converged(model):
    """GMRES stops early (before maxiter) only when the tolerance is reached."""
    if hasattr(model, "linear_solve_iteration_count"):
        iterations = model.linear_solve_iteration_count
    else:
        iterations = model.iteration_count
    return iterations, iterations < optimus.global_parameters.linalg.maxiter


def save_figures(figs, stem):
    """Save a list of matplotlib figures as PNG files in OUTPUT_DIR."""
    if not isinstance(figs, (list, tuple)):
        figs = [figs]
    for i, fig in enumerate(figs):
        fig.savefig(os.path.join(OUTPUT_DIR, "%s_%d.png" % (stem, i + 1)), dpi=110,
                    bbox_inches="tight")

## 1. The default database

In [ ]:
materials_table = optimus.material.common.get_excel_database(database="default")
print("Number of materials in the default database:", len(materials_table))
materials_table.head(5)

In [ ]:
names = ["water", "fat", "bone (cortical)"]
material_ext, material_fat, material_bone = optimus.material.load_material(names)
for material in (material_ext, material_fat, material_bone):
    material.print()

## 2. Derived quantities at 500 kHz

In [ ]:
frequency = 500e3
rows = []
for material in (material_ext, material_fat, material_bone):
    rows.append((material.name, material.density, material.speed_of_sound,
                 material.compute_wavelength(frequency) * 1e3,
                 material.compute_attenuation(frequency),
                 material.density * material.speed_of_sound / 1e6))
print("%-18s %9s %9s %12s %14s %10s" % ("material", "rho", "c", "lambda [mm]", "alpha [Np/m]", "Z [MRayl]"))
for r in rows:
    print("%-18s %9.1f %9.1f %12.3f %14.3f %10.3f" % r)

fig = plt.figure(figsize=(6, 3))
plt.bar([r[0] for r in rows], [r[5] for r in rows])
plt.ylabel("Z = rho c [MRayl]")
plt.title("Characteristic acoustic impedance")
save_figures(fig, NOTEBOOK_NAME)
plt.show()

## 3. Reflection and transmission at normal incidence

In [ ]:
def normal_incidence(m1, m2):
    z1 = m1.density * m1.speed_of_sound
    z2 = m2.density * m2.speed_of_sound
    r = (z2 - z1) / (z2 + z1)
    t = 1 + r
    return r, t, r**2 + (z1 / z2) * t**2

for m2 in (material_fat, material_bone):
    r, t, energy = normal_incidence(material_ext, m2)
    print("water -> %-16s R = %+.3f, T = %.3f, R^2 + (Z1/Z2) T^2 = %.6f" % (m2.name, r, t, energy))
    RESULTS["reflection_water_to_" + m2.name.replace(" ", "_")] = round(float(r), 4)

## 4. A new material

`create_material` needs a name, density and speed of sound; attenuation defaults to zero.
Properties can also be passed as an unpacked dictionary.

In [ ]:
material_user_1 = optimus.material.create_material(name="new-material-1", density=2500,
                                                   speed_of_sound=1500)
material_user_1.print()

properties = {"name": "gel-phantom", "density": 1040, "speed_of_sound": 1540,
              "attenuation_coeff_a": 5.0, "attenuation_pow_b": 1.1}
material_user_2 = optimus.material.create_material(**properties)
material_user_2.print()

## 5. Saving to the user-defined database (and undoing it)

`save_to_file=True` writes the material into `Material_database_user-defined.xlsx` inside the
installed `optimus/material` folder, so it is available in later sessions. This notebook keeps a
copy of the file first and puts it back at the end, so running the notebook leaves no trace.

In [ ]:
import shutil

user_database = os.path.join(os.path.dirname(optimus.material.common.__file__),
                             "Material_database_user-defined.xlsx")
backup = os.path.join(OUTPUT_DIR, "user_database_backup.xlsx")
shutil.copyfile(user_database, backup)
try:
    my_material = optimus.material.create_material(name="my-new-material", density=2500,
                                                   speed_of_sound=1492, save_to_file=True)
    reloaded = optimus.material.load_material("my-new-material")
    reloaded.print()
    saved_ok = (reloaded.density == 2500 and reloaded.speed_of_sound == 1492)

    try:
        optimus.material.create_material(name="my-new-material", density=1500,
                                         speed_of_sound=1000, save_to_file=True)
        duplicate_rejected = False
    except ValueError as error:
        print("Expected error:", error)
        duplicate_rejected = True
finally:
    shutil.copyfile(backup, user_database)
    os.remove(backup)
print("User database restored.")

## 6. Checks

In [ ]:
table = materials_table.set_index(("Tissue", "Name"))
water_row = table.loc["Water"] if "Water" in table.index else table.loc["water"]
check("database_values_loaded",
      np.isclose(material_ext.density, water_row[("Density (kg/m3)", "Average")])
      and np.isclose(material_ext.speed_of_sound, water_row[("Speed of Sound (m/s)", "Average")]),
      "water rho = %.1f, c = %.1f" % (material_ext.density, material_ext.speed_of_sound))

k = material_bone.compute_wavenumber(frequency)
check("wavenumber_definition",
      np.isclose(k.real, 2 * np.pi * frequency / material_bone.speed_of_sound)
      and np.isclose(material_bone.compute_wavelength(frequency), material_bone.speed_of_sound / frequency),
      "Re(k) = 2 pi f / c and lambda = c / f for bone")

check("attenuation_power_law",
      np.isclose(material_user_2.compute_attenuation(1e6), 5.0)
      and np.isclose(material_user_2.compute_attenuation(2e6), 5.0 * 2**1.1),
      "alpha(1 MHz) = a and alpha(2 MHz) = a 2^b")

energies = [normal_incidence(material_ext, m2)[2] for m2 in (material_fat, material_bone)]
check("energy_conserved", np.allclose(energies, 1.0), "R^2 + (Z1/Z2) T^2 = " + str(energies))

check("save_and_reload", saved_ok, "saved material reloaded with the same values")
check("duplicate_name_rejected", duplicate_rejected, "second save with the same name raised ValueError")
names_after = optimus.material.common.get_excel_database(database="user-defined", index_col=0)
check("user_database_restored",
      not names_after[("Tissue", "Name")].str.lower().isin(["my-new-material"]).any(),
      "my-new-material is no longer in the user database")

## Summary

Key numbers and check results, also saved as JSON in `OUTPUT_DIR` so CI and the assistant can read them.

In [ ]:
RESULTS["runtime_s"] = round(time.time() - START_TIME, 1)
with open(os.path.join(OUTPUT_DIR, NOTEBOOK_NAME + "_results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, sort_keys=True)
print(json.dumps(RESULTS, indent=2, sort_keys=True))